# CORnet face-inversion encoding GLM (the EEG analysis, applied to the model)

**Question.** The EEG analysis in `face_regressor_analysis.ipynb` showed the human brain
distinguishes **upright** from **upside-down (inverted)** faces: a continuous encoding GLM
regressing posterior 5–15 Hz power onto a face-inversion regressor (+0.5 inverted / −0.5 upright)
gives *more* power for inverted faces, group **t(7) ≈ 3.24, p ≈ 0.014**. A sibling RSA analysis
(`cornet_analysis.ipynb`) found **CORnet shows a null** — but RSA and the EEG GLM are different
statistics, so that null was hard to interpret.

This notebook removes that ambiguity by applying the **exact same encoding GLM** to CORnet-S, a
brain-like feed-forward model of the ventral visual stream (Krugliak & Clarke, 2022, is the EEG
paradigm; CORnet-S is Kubilius et al., 2019). We ask directly: *does a brain-like vision model
reproduce the inversion sensitivity the brain shows?* Same regressor, same GLM, same group t(7)
structure — so the CORnet result reads side-by-side with the EEG result.

### The mapping (EEG → CORnet)

| EEG (`face_regressor_analysis`) | CORnet port (this notebook) |
|---|---|
| 5–15 Hz Morlet band power per electrode, per time sample | Unit **activation** per CORnet unit, per frame (no wavelet — a model has no oscillations) |
| Posterior ROI = mean power over 17 occipital/parietal electrodes, z-scored | Layer ROI = mean activation over all units in a layer (V1/V2/V4/IT), z-scored |
| Per-electrode β + FDR (scalp map) | Per-unit β across subjects + FDR (count of significant units per layer; no topomap) |
| Face-inversion regressor +0.5/−0.5 (Hamming-convolved, continuous) | Same regressor; **event** mode = ±0.5 label per face frame, **dense** mode = the continuous Hamming version |
| One β per **person**, group t-test across 8 subjects → t(7) | One β per **subject's stimulus video**, group t-test across 8 → t(7) |
| β > 0 ⇒ more power for inverted | β > 0 ⇒ more activation for inverted |

### The caveat you must keep in mind

CORnet is **deterministic** — the model is byte-for-byte identical for every "subject." The only
thing that varies is each subject's own head-cam video (their particular sequence of upright/inverted
faces and backgrounds). So the group t-test here is a random-effects test **over stimulus
sequences**, not over biological replicates. It preserves the t(7) structure for a clean comparison,
but "significant across subjects" means *"consistent across these 8 independent video sequences,"*
not *"consistent across 8 brains."* This is stated again, in plain language, in the interpretation
at the bottom.

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Import the shared EEG helpers (eeg_utils.py, in src/eda) and the CORnet
# plumbing (cornet_utils.py, alongside this notebook). The shim makes the
# notebook runnable from either the repo root or its own folder.
_HERE = Path.cwd()
for _p in [_HERE, _HERE / "src" / "eda", _HERE.parent, _HERE / "nn_models",
           _HERE.parent / "nn_models"]:
    if (_p / "eeg_utils.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
    if (_p / "cornet_utils.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import eeg_utils as eu          # noqa: E402
import cornet_utils as cu       # noqa: E402  (heavy torch/cornet imports are lazy)

warnings.filterwarnings("ignore", category=RuntimeWarning)  # zero-variance z-scores
print("eeg_utils and cornet_utils imported OK")

## Configuration

Every methodological choice is a **switch**, not a hard-coded decision, so the whole analysis can be
re-run under a different setting by editing one line here.

- **`SAMPLING_MODE`** — `"event"` (default, fast): read cached CORnet activations at each face
  onset; the regressor is the ±0.5 label per event. `"dense"` (slow, re-runs the model): decode the
  video at `DENSE_FPS`, run CORnet on every frame, use the continuous Hamming-convolved regressor.
- **`INPUT_MODE`** — `"full_frame"` (default): CORnet sees the whole letterboxed scene.
  `"facecrop"`: CORnet sees a face-centred crop. Event mode picks the matching cached `.npz`; dense
  mode applies the Haar crop live.
- **`GROUP_MODE`** — `"rfx"` (default): one β per subject, then a one-sample t-test across the 8
  (mirrors the EEG t(7)). `"pooled"`: one GLM over every subject's samples combined — more power but
  pseudoreplicated, reported only as a secondary robustness check.

In [ ]:
# --- switches (see the markdown above) ---
SAMPLING_MODE = "event"        # "event" | "dense"
INPUT_MODE    = "full_frame"   # "full_frame" | "facecrop"
GROUP_MODE    = "rfx"          # "rfx" | "pooled"

# --- fixed analysis parameters ---
LAYERS      = cu.LAYERS                 # ["V1", "V2", "V4", "IT"], early -> late
SUBJECTS    = list(eu.SUBJECTS)         # 1..8
OUTLIER_SD  = 4.0                       # authors' 4-SD outlier cut (posterior_betas / elec_glm)
RANDOM_SEED = 42

# dense-mode only (ignored in event mode)
DENSE_FPS      = 5                      # frames/s to decode + run through CORnet (slow!)
DENSE_SUBJECTS = SUBJECTS               # e.g. [1] to validate the re-run path on one subject first
DENSE_MAX_UNITS = 4096                  # subsample IT units for the dense per-unit map (bound memory)

# The EEG posterior-ROI headline from face_regressor_analysis.ipynb, used as the
# reference in the comparison section. Loaded from faithful_port_results.npz when
# present (it is git-ignored / kept locally); these constants are the fallback.
EEG_REF_T, EEG_REF_P = 3.24, 0.014

OUT = eu.DATA_ROOT.parent / "cornet_analysis_outputs"
rng = np.random.default_rng(RANDOM_SEED)

print(f"mode: sampling={SAMPLING_MODE}  input={INPUT_MODE}  group={GROUP_MODE}")
print(f"layers={LAYERS}  subjects={SUBJECTS}")
print(f"cached-activation dir: {OUT}")

## 1. Signal extraction

`get_layer_signals(sub, ...)` returns, per layer, the two things the GLM needs:

- a **scalar ROI signal per frame** — the mean activation across all units in the layer (the model
  analog of the EEG posterior ROI, which averages across electrodes), and
- the aligned **inversion regressor** and **labels**,

plus, for the per-unit map, the **full per-unit activation matrix** (`units`, shape
`n_frames × n_units`).

- **Event mode** just reads the cached `.npz` (`event_{layer}` = activations at each face onset,
  `event_labels` = UP/IN); the regressor is the ±0.5 label. No model is run. *Because the regressor
  takes only two values here, the GLM slope is proportional to mean(inverted) − mean(upright) — i.e.
  exactly the inversion contrast.*
- **Dense mode** decodes the subject's video at `DENSE_FPS`, runs CORnet per frame, and builds the
  continuous regressor with `eu.build_face_regressors` after `raw.resample(DENSE_FPS)`. The one
  subtlety is **alignment**: CORnet frames are indexed in *video* time, the regressor in *EEG* time,
  offset by the video-start trigger `t1`, so frame *i* (video time `i/DENSE_FPS`) pairs with the
  regressor sample at EEG index `round((i/DENSE_FPS + t1) * DENSE_FPS)`. To bound memory, the dense
  per-unit matrix is kept for **IT only**, subsampled to `DENSE_MAX_UNITS` units.

In [ ]:
def _event_signals(sub, input_mode):
    "Event mode: load cached per-event activations; regressor = +/-0.5 label."
    suffix = "_facecrop" if input_mode == "facecrop" else ""
    path = OUT / f"sub{sub}_cornet{suffix}.npz"
    d = np.load(path, allow_pickle=True)
    labels = np.array([str(s).strip().upper() for s in d["event_labels"]])
    reg = np.where(labels == "IN", 0.5, -0.5).astype(float)
    per_layer = {}
    for L in LAYERS:
        A = np.asarray(d[f"event_{L}"], dtype=float)     # (n_events, n_units)
        per_layer[L] = {"roi": A.mean(axis=1), "units": A}
    return per_layer, reg, labels


def _dense_signals(sub, input_mode, dense_fps):
    "Dense mode: decode video @ dense_fps, run CORnet per frame, continuous regressor."
    raw = eu.load_eeg(sub)
    raw.resample(dense_fps)                               # build_face_regressors is rate-agnostic
    r = eu.regressor_embedded_corr(raw)
    print(f"  sub{sub}: regressor vs embedded FaceInversion r={r:.3f} (expect ~1.0)")
    regs = eu.build_face_regressors(raw)
    inv = regs["face_inversion"]
    t1 = eu.get_t1(sub, raw)

    model, activations, handles = cu.load_model_and_hooks(LAYERS)
    cap, fps, n_frames = cu.open_video(sub)
    video_dur = n_frames / fps
    n_steps = int(video_dur * dense_fps)

    roi = {L: [] for L in LAYERS}
    it_units = []
    reg_vals = []
    try:
        for i in range(n_steps):
            t_v = i / dense_fps
            eeg_idx = int(round((t_v + t1) * dense_fps))
            if not (0 <= eeg_idx < inv.size):
                continue
            frame = cu.grab_frame(cap, t_v, fps)
            if frame is None:
                continue
            if input_mode == "facecrop":
                frame = cu.crop_to_face(frame, cu.detect_face_bbox(frame))
            feats = cu.frame_to_features(frame, model, activations, LAYERS)
            for L in LAYERS:
                roi[L].append(float(feats[L].mean()))
            it_units.append(feats["IT"])
            reg_vals.append(inv[eeg_idx])
    finally:
        cap.release()
        for h in handles:
            h.remove()

    reg = np.asarray(reg_vals, float)
    IT = np.asarray(it_units, float)                     # (n_frames, n_IT_units)
    if IT.shape[1] > DENSE_MAX_UNITS:                    # subsample to bound memory
        pick = rng.choice(IT.shape[1], DENSE_MAX_UNITS, replace=False)
        IT = IT[:, pick]
    per_layer = {L: {"roi": np.asarray(roi[L], float),
                     "units": IT if L == "IT" else None} for L in LAYERS}
    labels = np.where(reg > 0, "IN", "UP")               # nominal, for reporting only
    return per_layer, reg, labels


def get_layer_signals(sub, sampling_mode=None, input_mode=None, dense_fps=None):
    "Dispatch to the event (cached) or dense (re-run) extractor."
    sampling_mode = SAMPLING_MODE if sampling_mode is None else sampling_mode
    input_mode = INPUT_MODE if input_mode is None else input_mode
    dense_fps = DENSE_FPS if dense_fps is None else dense_fps
    if sampling_mode == "event":
        return _event_signals(sub, input_mode)
    if sampling_mode == "dense":
        return _dense_signals(sub, input_mode, dense_fps)
    raise ValueError(f"unknown SAMPLING_MODE {sampling_mode!r}")

## 2. The GLM

`fit_betas` is the encoding GLM, the direct analog of the EEG `posterior_betas` / `elec_glm`
functions: **z-score the signal, drop 4-SD outlier frames, then take the OLS slope of signal on the
inversion regressor.** It is vectorized over columns so the same code serves both the single ROI
signal and the whole per-unit matrix (each unit is a column, with its own outlier mask), and it also
returns each slope's two-sided p-value for the per-unit FDR. `bh_fdr` is ported verbatim from
`face_regressor_analysis.ipynb`.

In [ ]:
def bh_fdr(p, q=0.05):
    "Benjamini-Hochberg FDR mask (verbatim from face_regressor_analysis.ipynb)."
    p = np.asarray(p); n = len(p)
    ranked = np.sort(p)
    below = ranked <= q * (np.arange(1, n + 1) / n)
    if not below.any():
        return np.zeros(n, bool)
    return p <= ranked[int(np.max(np.where(below)))]


def fit_betas(signal, regressor, outlier_sd=OUTLIER_SD):
    """Encoding-GLM slope of (z-scored signal) ~ regressor, vectorized over columns.

    signal: (n_frames,) scalar ROI, or (n_frames, n_units) per-unit matrix.
    Z-scores each column over frames, drops frames >outlier_sd from that column's
    mean (per-column mask, matching posterior_betas / elec_glm), then OLS slope and
    its two-sided p-value. Returns (slope, p): scalars if signal is 1-D, else arrays.
    Zero-variance columns yield NaN (handled by nan-aware aggregation downstream).
    """
    x = np.asarray(regressor, float)
    Y = np.asarray(signal, float)
    one_d = Y.ndim == 1
    if one_d:
        Y = Y[:, None]
    Z = stats.zscore(Y, axis=0)                          # per-column z-score over frames
    M = (np.abs(Z) <= outlier_sd).astype(float)          # per-column keep mask
    xb = x[:, None]
    n = M.sum(0)
    xbar = (M * xb).sum(0) / n
    ybar = (M * Z).sum(0) / n
    Sxx = (M * xb**2).sum(0) - n * xbar**2
    Sxy = (M * xb * Z).sum(0) - n * xbar * ybar
    Syy = (M * Z**2).sum(0) - n * ybar**2
    slope = Sxy / Sxx
    df = n - 2
    resid = np.clip(Syy - slope * Sxy, 0, None)
    se = np.sqrt((resid / df) / Sxx)
    t = slope / se
    p = 2 * stats.t.sf(np.abs(t), df)
    if one_d:
        return float(slope[0]), float(p[0])
    return slope, p


def rfx_ttest(betas):
    "Group random-effects: one-sample t-test of per-subject betas vs 0 -> (t, p)."
    b = np.asarray(betas, float)
    b = b[np.isfinite(b)]
    t, p = stats.ttest_1samp(b, 0.0)
    return float(t), float(p)

## 3. Run the GLM for every subject and layer

For each subject we extract the signals once, then per layer take (a) the **scalar-ROI β** — the
headline number, one per subject — and (b) the **per-unit β vector**. Because CORnet is the same
model for everyone, unit *j* of a layer is the *same feature detector* in every subject, so per-unit
βs are directly comparable across subjects (exactly as scalp electrodes are in the EEG map). We keep
the pooled ROI signal/regressor too, so the `"pooled"` group option can fit a single combined GLM.

In [ ]:
roi_betas   = {L: [] for L in LAYERS}   # per-subject scalar-ROI beta  -> (n_sub,)
unit_betas  = {L: [] for L in LAYERS}   # per-subject per-unit beta vector (where available)
pooled_sig  = {L: [] for L in LAYERS}   # concatenated ROI signal (for GROUP_MODE="pooled")
pooled_reg  = []
subs_used   = DENSE_SUBJECTS if SAMPLING_MODE == "dense" else SUBJECTS

for sub in subs_used:
    per_layer, reg, labels = get_layer_signals(sub)
    n_in, n_up = int((labels == "IN").sum()), int((labels == "UP").sum())
    pooled_reg.append(reg)
    msg = [f"sub{sub}: frames={reg.size} (IN={n_in}, UP={n_up})"]
    for L in LAYERS:
        roi = per_layer[L]["roi"]
        b, _ = fit_betas(roi, reg)
        roi_betas[L].append(b)
        pooled_sig[L].append(roi)
        if per_layer[L]["units"] is not None:
            ub, _ = fit_betas(per_layer[L]["units"], reg)
            unit_betas[L].append(ub)
        msg.append(f"{L} beta={b:+.3f}")
    print("  ".join(msg))

roi_betas  = {L: np.array(roi_betas[L]) for L in LAYERS}
pooled_reg = np.concatenate(pooled_reg)
pooled_sig = {L: np.concatenate(pooled_sig[L]) for L in LAYERS}

### Group statistics per layer

`"rfx"` runs a one-sample t-test on the 8 per-subject βs (the EEG t(7) analog). `"pooled"` fits one
GLM over all subjects' frames stacked together and reports that slope's own t/p. The per-unit map is
always computed the random-effects way — mean β and a one-sample t-test **across subjects** per unit,
then FDR — because that is the true analog of the EEG per-electrode scalp map.

In [ ]:
layer_stats = {}                     # L -> (beta_summary, t, p, direction)
for L in LAYERS:
    if GROUP_MODE == "rfx":
        b = roi_betas[L]
        t, p = rfx_ttest(b)
        summary = np.nanmean(b)
    elif GROUP_MODE == "pooled":
        # One GLM over every subject's frames stacked together. Same z-score + 4-SD
        # cut as fit_betas, run through linregress so the slope's t and p come out
        # directly (fit_betas returns only slope + p, and here we also want t).
        y = stats.zscore(pooled_sig[L])
        keep = np.abs(y) <= OUTLIER_SD
        lr = stats.linregress(pooled_reg[keep], y[keep])
        summary, t, p = lr.slope, lr.slope / lr.stderr, lr.pvalue
    else:
        raise ValueError(f"unknown GROUP_MODE {GROUP_MODE!r}")
    layer_stats[L] = (summary, t, p, "inverted>upright" if summary > 0 else "upright>inverted")

print(f"GROUP RESULTS  (mode={GROUP_MODE}; positive beta => more activation for INVERTED)\n")
for L in LAYERS:
    s, t, p, d = layer_stats[L]
    star = " *" if p < 0.05 else ""
    print(f"  {L:>3}:  beta={s:+.4f}  t={t:+.2f}  p={p:.4f}  ({d}){star}")

# Per-unit random-effects map (needs >=2 subjects with a per-unit matrix)
unit_map = {}                       # L -> dict(mean_beta, p, fdr_mask) or None
for L in LAYERS:
    if len(unit_betas[L]) >= 2:
        B = np.vstack(unit_betas[L])            # (n_sub, n_units), same units across subjects
        mean_b = np.nanmean(B, axis=0)
        t_u, p_u = stats.ttest_1samp(B, 0.0, axis=0, nan_policy="omit")
        p_u = np.asarray(p_u, float)
        fdr = bh_fdr(np.nan_to_num(p_u, nan=1.0))
        unit_map[L] = {"mean_beta": mean_b, "p": p_u, "fdr": fdr}
        print(f"  {L}: per-unit map {B.shape[1]} units, "
              f"{int(fdr.sum())} survive FDR (q=0.05)")
    else:
        unit_map[L] = None

## 4. Results

**Left:** the headline — per-layer ROI β (V1 → IT). In `"rfx"` each subject is a dot and the bar is
the group mean; the annotation is the t/p. A bar reliably above zero means CORnet's activation in
that layer is higher for inverted faces, the same direction the brain shows. **Right:** the per-unit
map — the distribution of per-unit βs per layer with the count surviving FDR (the model analog of
the EEG scalp map; there is no topomap because CORnet units are not laid out in space).

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.8))

# --- left: per-layer ROI beta ---
xs = np.arange(len(LAYERS))
means = [layer_stats[L][0] for L in LAYERS]
axL.axhline(0, color="grey", lw=0.8)
axL.bar(xs, means, color="#4C72B0", alpha=0.75, width=0.6, zorder=2)
if GROUP_MODE == "rfx":
    for i, L in enumerate(LAYERS):
        b = roi_betas[L]
        axL.scatter(np.full_like(b, i) + rng.uniform(-0.12, 0.12, b.size), b,
                    color="k", s=22, zorder=3, alpha=0.8)
for i, L in enumerate(LAYERS):
    _, t, p, _ = layer_stats[L]
    axL.annotate(f"t={t:.2f}\np={p:.3f}", (i, means[i]),
                 textcoords="offset points", xytext=(0, 8 if means[i] >= 0 else -22),
                 ha="center", fontsize=9)
axL.set_xticks(xs); axL.set_xticklabels(LAYERS)
axL.set_ylabel("inversion beta (z-scored activation)")
axL.set_title(f"Per-layer face-inversion beta  ({GROUP_MODE}, {SAMPLING_MODE}/{INPUT_MODE})")

# --- right: per-unit beta distributions + FDR count ---
have = [L for L in LAYERS if unit_map[L] is not None]
if have:
    data = [unit_map[L]["mean_beta"][np.isfinite(unit_map[L]["mean_beta"])] for L in have]
    parts = axR.violinplot(data, showextrema=False)
    for pc in parts["bodies"]:
        pc.set_facecolor("#55A868"); pc.set_alpha(0.6)
    axR.axhline(0, color="grey", lw=0.8)
    axR.set_xticks(np.arange(1, len(have) + 1))
    axR.set_xticklabels([f"{L}\n{int(unit_map[L]['fdr'].sum())} sig" for L in have])
    axR.set_ylabel("per-unit mean beta")
    axR.set_title("Per-unit inversion beta (FDR-significant count)")
else:
    axR.text(0.5, 0.5, "per-unit map needs >=2 subjects\nwith a unit matrix",
             ha="center", va="center", transform=axR.transAxes)
    axR.axis("off")

fig.tight_layout()
plt.show()

## 5. Comparison to the brain

Put the CORnet result next to the EEG posterior-ROI headline. We try to load the actual per-subject
EEG betas from `faithful_port_results.npz` (kept locally, git-ignored); if that file or an expected
key is missing we fall back to the published headline **t(7) = 3.24, p = 0.014**. The point of the
plot is the direct visual comparison: does any CORnet layer land in the same above-zero,
`p < 0.05` territory the brain occupies?

In [ ]:
ref_t, ref_p, ref_src = EEG_REF_T, EEG_REF_P, "fallback constants"
ref_betas = None
ref_path = OUT / "faithful_port_results.npz"
if ref_path.exists():
    ref = np.load(ref_path, allow_pickle=True)
    print("faithful_port_results.npz keys:", list(ref.files))
    # Look for a 1-D length-8 posterior-beta array under a few likely names.
    for k in ("allpossub", "posterior_betas", "pos_betas", "betas", "allpossub_reg"):
        if k in ref.files and np.ndim(ref[k]) == 1 and np.asarray(ref[k]).size == len(SUBJECTS):
            ref_betas = np.asarray(ref[k], float)
            ref_t, ref_p = rfx_ttest(ref_betas)
            ref_src = f"{ref_path.name}[{k}]"
            break
    else:
        # Otherwise try scalar t / p entries.
        for tk, pk in (("t_pos", "p_pos"), ("t", "p")):
            if tk in ref.files and pk in ref.files:
                ref_t, ref_p = float(ref[tk]), float(ref[pk])
                ref_src = f"{ref_path.name}[{tk},{pk}]"
                break
else:
    print(f"{ref_path.name} not found -> using fallback constants")
print(f"EEG reference: t(7)={ref_t:.2f}, p={ref_p:.3f}  (source: {ref_src})")

fig, ax = plt.subplots(figsize=(8, 4.6))
xs = np.arange(len(LAYERS))
cor_t = [layer_stats[L][1] for L in LAYERS]
colors = ["#C44E52" if layer_stats[L][2] < 0.05 else "#8C8C8C" for L in LAYERS]
ax.bar(xs, cor_t, color=colors, alpha=0.8, width=0.6, label="CORnet layer (red = p<0.05)")
ax.axhline(0, color="grey", lw=0.8)
ax.axhline(ref_t, color="#4C72B0", lw=2, ls="--",
           label=f"EEG posterior ROI  t(7)={ref_t:.2f}, p={ref_p:.3f}")
# two-sided t critical value for df=7, p=0.05, as a reference band
tcrit = stats.t.ppf(0.975, len(SUBJECTS) - 1)
ax.axhline(tcrit, color="k", lw=0.6, ls=":")
ax.axhline(-tcrit, color="k", lw=0.6, ls=":")
ax.set_xticks(xs); ax.set_xticklabels(LAYERS)
ax.set_ylabel("group t-statistic")
ax.set_title("Face-inversion sensitivity: CORnet layers vs the brain")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 6. Interpretation

### 6.0 What this notebook did, in one plain sentence

We took the *exact* test that found a face-inversion effect in the human EEG and re-ran it on a
brain-like computer-vision model, asking whether the model's internal activity also cares whether a
face is upright or upside-down.

### 6.1 A short glossary (for readers new to the modelling side)

- **CORnet-S** is a neural network trained to recognise objects in images. Its layers **V1, V2, V4,
  IT** are named after real stages of the primate ventral visual stream and, when the network is
  fed an image, they respond in roughly the order and manner those brain areas do — which is why it
  is used as a *model of vision*, not just a classifier.
- A **unit** is one artificial neuron; an **activation** is the number it outputs for a given image
  (how strongly its particular feature is present). A layer has thousands to hundreds of thousands
  of units.
- **Why there is no wavelet here.** In the EEG we measured 5–15 Hz *power* — the strength of a brain
  rhythm — because neurons oscillate. A network has no oscillations, so the natural equivalent of
  "how active is this region right now" is simply the **activation** itself. That is the one, and
  only, deliberate departure from the EEG recipe; everything else (the ±0.5 regressor, the z-score,
  the 4-SD cut, the OLS slope, the group t-test) is identical.
- **What β means.** β is the slope of activation on the inversion regressor. **β > 0 means the layer
  is more active for inverted than upright faces**; β ≈ 0 means it does not distinguish them. The
  group t-test asks whether β is reliably non-zero across the 8 subjects' videos.

### 6.2 The one caveat that changes how you read significance

CORnet is **deterministic and identical for every subject**. In the EEG, eight *different brains*
each gave a β, and a significant t(7) meant the effect generalised across *people*. Here the network
never changes — what changes is each subject's **video**, i.e. their particular sequence of faces
and cluttered backgrounds. So a significant t(7) means the effect is **consistent across 8
independent stimulus sequences**, which is a real and meaningful random-effects claim *over stimuli*,
but it is **not** a claim about variability across models or brains. Read "significant" as
"robust to which video you show it," not "robust across individuals."

### 6.3 How to read the actual result

- **A layer with β > 0 and p < 0.05** (red bar in §5) means CORnet reproduces the brain's
  direction: more internal response to inverted faces. If this appears and *strengthens toward IT*,
  the model's higher, more face-like stages behave like the posterior EEG signal — a **convergence**
  with the brain.
- **All layers near zero / n.s.** would mean the model does **not** reproduce the inversion effect
  under this test — a **dissociation**, and (crucially) one measured with the *same* statistic that
  is significant in the brain, so it can no longer be dismissed as an RSA-vs-GLM artefact. That
  makes the earlier RSA null in `cornet_analysis.ipynb` interpretable rather than mysterious.
- The **per-unit map** (§4, right) refines this: even if the layer-average ROI is flat, a subset of
  units might carry a reliable inversion signal (surviving FDR). Many significant units with mixed
  signs can average to ~0 at the ROI level — worth checking before concluding "no effect."

### 6.4 Honest limitations

- **Event mode is a two-level contrast.** With the ±0.5 regressor the ROI slope is essentially
  mean(inverted) − mean(upright) over a few dozen onset frames per subject; it is a clean contrast
  but low-N. **Dense mode** (continuous regressor, many frames) is the stronger test — validate it
  on one subject (`DENSE_SUBJECTS=[1]`) before trusting the group.
- **The stimulus is not a clean face.** These are mobile head-cam frames of printed portraits amid
  posters and text; `"facecrop"` reduces the clutter but the Haar detector is imperfect, especially
  for inverted faces. Comparing `full_frame` vs `facecrop` is the honest robustness check.
- **CORnet is feed-forward.** It has no recurrence or expertise-tuned face machinery of the kind
  usually invoked to explain the human inversion effect, so a null here is a *plausible* outcome,
  not a bug — and is itself an informative result about what this class of model does and does not
  capture.